# ☕🚲 Ćwiczenie 5: Spark SQL i DataFrames (Sieć kawiarni) + Window Functions (Rowery miejskie)

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 5 (Studia stacjonarne, Semestr Zimowy 2026/2027)
**Szacowany czas:** ⏱️ ~180 minut (2× ~90 minut)
**Poziom trudności:** ⭐⭐–⭐⭐⭐ (Średni)

To laboratorium łączy dwa powiązane tematy w jednym notebooku:

- **Część A — Spark SQL i DataFrames** na syntetycznych transakcjach sieci kawiarni ☕
- **Część B — Window Functions** na syntetycznych przejazdach wypożyczalni rowerów miejskich 🚲

## 📚 Cele nauki

Po ukończeniu tego notebooka będziesz potrafić:

**Część A — Spark SQL i DataFrames:**
1. Rejestrować tymczasowe widoki i pisać zapytania `spark.sql()`
2. Wykonywać agregacje (`groupBy`, `agg`) i porównywać je z odpowiednikiem SQL
3. Łączyć dane transakcyjne z tabelą wymiarową (`join`, `broadcast`)
4. Analizować dane czasowe (godzina, miesiąc) i liczyć wzrosty miesiąc do miesiąca
5. Wskazywać najlepiej sprzedające się produkty i metody płatności per sklep/miasto

**Część B — Window Functions:**
6. Definiować **window functions** przy pomocy `Window.partitionBy()` i `orderBy()`
7. Liczyć **rankingi** (`rank`, `row_number`) oraz **sumy narastające** (`rowsBetween`)
8. Używać `lag()`/`lead()` do analiz sekwencyjnych (poprzedni/następny przejazd)
9. Czytać plan zapytania (`.explain()`) i rozpoznawać shuffle (`Exchange`)
10. Porównać `repartition()` i `coalesce()` pod kątem wydajności

## 📋 Wymagania wstępne
- Podstawy PySpark (RDD, transformacje, akcje) — Wykład 4
- Podstawy Spark SQL i DataFrame API — Wykład 5

---
> 💡 **Tip:** Uruchamiaj każdą komórkę przez `Shift+Enter` lub przycisk ▶️.
> Jeśli utkniesz, sprawdź komórki z podpowiedziami (💡 Hint) albo zapytaj prowadzącego!
> Notebook ma dwie niezależne części — każda generuje własny zbiór danych i może być uruchamiana samodzielnie.


---

# 🅰️ Część A: Spark SQL i DataFrames — Sieć kawiarni ☕

Analiza ~100 000 transakcji z sieci kawiarni działającej w 15 lokalach w różnych miastach Polski.


In [ ]:
# @title 🔧 Setup — Run this cell first!

!pip install -q pyspark

import random
import datetime as dt

import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("CW05-SiecKawiarni")
    .master("local[*]")
    .getOrCreate()
)

random.seed(42)
np.random.seed(42)

print(f"✅ Setup complete! Spark {spark.version} gotowy do parzenia danych ☕🚀")

## 📖 Szybki refresher

Pracujemy dziś z danymi transakcyjnymi **sieci kawiarni** — 15 lokali w kilku miastach Polski, ~100 000 transakcji.

| Koncept | Opis | Przykład |
|---------|------|---------|
| `createOrReplaceTempView` | Rejestruje DataFrame jako tabelę SQL | `df.createOrReplaceTempView("transakcje")` |
| `spark.sql()` | Wykonuje zapytanie SQL na widokach | `spark.sql("SELECT * FROM transakcje")` |
| `groupBy().agg()` | Agregacje w DataFrame API | `df.groupBy("miasto").agg(F.sum("cena"))` |
| `broadcast()` | Wymusza kopiowanie małej tabeli do każdego executora | `df.join(F.broadcast(mala_tabela), "klucz")` |
| `Window` | Operacje analityczne bez redukcji liczby wierszy | `F.rank().over(Window.partitionBy(...))` |

In [ ]:
# @title 📊 Generujemy dane — transakcje sieci kawiarni

MIASTA = ["Warszawa", "Krakow", "Wroclaw", "Poznan", "Gdansk"]
REGIONY = {
    "Warszawa": "Mazowieckie",
    "Krakow": "Malopolskie",
    "Wroclaw": "Dolnoslaskie",
    "Poznan": "Wielkopolskie",
    "Gdansk": "Pomorskie",
}

# 3 sklepy na miasto = 15 sklepów łącznie
sklepy_lista = []
sklep_do_miasta = {}
for i, miasto in enumerate(MIASTA):
    for j in range(3):
        sklep_id = f"SKL{i * 3 + j + 1:02d}"
        sklep_do_miasta[sklep_id] = miasto
        sklepy_lista.append({
            "sklep": sklep_id,
            "miasto": miasto,
            "region": REGIONY[miasto],
            "powierzchnia_m2": random.randint(35, 120),
            "data_otwarcia": dt.date(2020, 1, 1) + dt.timedelta(days=random.randint(0, 1800)),
        })

sklepy_pdf = pd.DataFrame(sklepy_lista)
sklepy = spark.createDataFrame(sklepy_pdf)
sklepy.show()

In [ ]:
# @title 📊 Generujemy ~100 000 transakcji

PRODUKTY = [
    # (produkt, kategoria, cena_bazowa)
    ("Espresso", "kawa", 8.0),
    ("Cappuccino", "kawa", 12.0),
    ("Latte", "kawa", 13.0),
    ("Americano", "kawa", 9.0),
    ("Flat White", "kawa", 14.0),
    ("Herbata czarna", "herbata", 7.0),
    ("Herbata zielona", "herbata", 7.5),
    ("Herbata owocowa", "herbata", 8.0),
    ("Croissant", "przekaska", 9.0),
    ("Sernik", "przekaska", 14.0),
    ("Muffin", "przekaska", 10.0),
    ("Kanapka", "przekaska", 15.0),
    ("Woda", "inne", 6.0),
    ("Sok", "inne", 9.0),
    ("Czekolada na goraco", "inne", 12.0),
]

METODY_PLATNOSCI = ["karta", "gotowka", "blik"]
METODY_WAGI = [0.55, 0.20, 0.25]

N_TRANSAKCJI = 100_000
START_DATA = dt.datetime(2025, 11, 1)

sklepy_ids = sklepy_pdf["sklep"].tolist()

rows = []
for tid in range(1, N_TRANSAKCJI + 1):
    sklep = random.choice(sklepy_ids)
    miasto = sklep_do_miasta[sklep]
    produkt, kategoria, cena_bazowa = random.choice(PRODUKTY)

    # więcej ruchu rano (7-10) i popoludniu (12-15) niż wieczorem
    godzina = int(np.clip(np.random.normal(loc=12, scale=3.5), 6, 21))
    dzien_offset = random.randint(0, 364)
    minuta = random.randint(0, 59)
    data_czas = START_DATA + dt.timedelta(days=dzien_offset, hours=godzina, minutes=minuta)

    cena = round(cena_bazowa + np.random.normal(0, 0.5), 2)
    ilosc = random.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0]
    metoda_platnosci = random.choices(METODY_PLATNOSCI, weights=METODY_WAGI)[0]

    rows.append((
        tid, data_czas, sklep, miasto, produkt, kategoria,
        max(cena, 1.0), ilosc, metoda_platnosci,
    ))

kolumny = [
    "transakcja_id", "data_czas", "sklep", "miasto", "produkt",
    "kategoria", "cena", "ilosc", "metoda_platnosci",
]
transakcje_pdf = pd.DataFrame(rows, columns=kolumny)

transakcje = spark.createDataFrame(transakcje_pdf)
transakcje = transakcje.withColumn("data_czas", F.col("data_czas").cast("timestamp"))

print(f"📊 Liczba transakcji: {transakcje.count()}")
transakcje.printSchema()
transakcje.show(5)

## 🎓 Guided Example: przychód per miasto

**Zadanie:** Oblicz łączny przychód (`cena * ilosc`) w podziale na miasto, posortowany malejąco.

Zobaczysz to samo zadanie napisane **dwoma sposobami** — Spark SQL i DataFrame API — żeby zobaczyć, że dają identyczny wynik.

In [ ]:
# @title 🎓 Guided Example — Run and study this

# Krok 1: rejestrujemy tymczasowy widok
transakcje.createOrReplaceTempView("transakcje")
sklepy.createOrReplaceTempView("sklepy")

# Krok 2: to samo zadanie — Spark SQL
przychod_sql = spark.sql("""
    SELECT miasto, ROUND(SUM(cena * ilosc), 2) AS przychod
    FROM transakcje
    GROUP BY miasto
    ORDER BY przychod DESC
""")

print("💬 Wynik SQL:")
przychod_sql.show()

# Krok 3: to samo zadanie — DataFrame API
przychod_api = (
    transakcje
    .withColumn("wartosc", F.col("cena") * F.col("ilosc"))
    .groupBy("miasto")
    .agg(F.round(F.sum("wartosc"), 2).alias("przychod"))
    .orderBy(F.col("przychod").desc())
)

print("🐍 Wynik DataFrame API:")
przychod_api.show()

---
## ✏️ Ćwiczenie 1: Najlepiej sprzedający się produkt per sklep

**Trudność:** ⭐⭐

Dla każdego sklepu znajdź produkt, który wygenerował **najwyższy przychód** (`cena * ilosc` zsumowane).

Wynikowy DataFrame `najlepszy_produkt` powinien mieć kolumny: `sklep`, `produkt`, `przychod` — dokładnie **jeden wiersz na sklep**.

**Oczekiwany rezultat:** DataFrame z 15 wierszami (po jednym na sklep).

> 💡 **Hint:** Policz przychód per `sklep` + `produkt`, a potem użyj `Window.partitionBy("sklep").orderBy(...)` z `F.row_number()`, żeby wybrać wiersz numer 1 w każdej grupie.

In [ ]:
# ✏️ Ćwiczenie 1: Najlepiej sprzedający się produkt per sklep

# TODO: policz przychód per sklep + produkt
przychod_produkt_sklep = ...  # Replace ... with your code

# TODO: zdefiniuj okno partycjonowane po sklepie, posortowane malejąco po przychodzie
okno_sklep = ...  # Replace ... with your code

# TODO: dodaj kolumnę z rankingiem i zostaw tylko ranga == 1
najlepszy_produkt = ...  # Replace ... with your code

najlepszy_produkt.show()

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 1 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: liczymy przychód (cena * ilosc) zsumowany per sklep + produkt
przychod_produkt_sklep = (
    transakcje
    .withColumn("wartosc", F.col("cena") * F.col("ilosc"))
    .groupBy("sklep", "produkt")
    .agg(F.sum("wartosc").alias("przychod"))
)

# Krok 2: okno partycjonowane po sklepie, posortowane malejąco wg przychodu danego produktu
okno_sklep = Window.partitionBy("sklep").orderBy(F.desc("przychod"))

# Krok 3: nadajemy numer wiersza w ramach każdego sklepu i zostawiamy tylko numer 1 (lidera)
najlepszy_produkt = (
    przychod_produkt_sklep
    .withColumn("ranking", F.row_number().over(okno_sklep))
    .filter(F.col("ranking") == 1)
    .select("sklep", "produkt", "przychod")
)

najlepszy_produkt.show()


In [ ]:
# @title ✅ Test — Ćwiczenie 1 (Run to check your answer)

try:
    assert najlepszy_produkt is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    liczba_wierszy = najlepszy_produkt.count()
    assert liczba_wierszy == 15, f"❌ Oczekiwano 15 wierszy (po jednym na sklep), otrzymano {liczba_wierszy}"
    kolumny_wyniku = set(najlepszy_produkt.columns)
    assert {"sklep", "produkt", "przychod"}.issubset(kolumny_wyniku), "❌ Brakuje wymaganych kolumn"
    liczba_unikalnych_sklepow = najlepszy_produkt.select("sklep").distinct().count()
    assert liczba_unikalnych_sklepow == 15, "❌ Każdy sklep powinien wystąpić dokładnie raz"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 2: Przychód per kategoria i godzina dnia

**Trudność:** ⭐⭐

Chcemy zobaczyć, o której porze dnia sprzedaje się najwięcej poszczególnych kategorii produktów.

1. Dodaj kolumnę `godzina` wyciągniętą z `data_czas` (użyj `F.hour()`)
2. Stwórz tabelę przestawną (`pivot`) — wiersze to `godzina`, kolumny to `kategoria`, wartości to suma przychodu

**Oczekiwany rezultat:** DataFrame z kolumną `godzina` oraz osobną kolumną dla każdej kategorii (`kawa`, `herbata`, `przekaska`, `inne`).

> 💡 **Hint:** `.groupBy("godzina").pivot("kategoria").agg(F.sum(...))` — pamiętaj o dodaniu kolumny `wartosc` przed pivotem!

In [ ]:
# ✏️ Ćwiczenie 2: Przychód per kategoria i godzina dnia

# TODO: dodaj kolumnę 'godzina' oraz 'wartosc' (cena * ilosc)
transakcje_z_godzina = ...  # Replace ... with your code

# TODO: pivot — wiersze: godzina, kolumny: kategoria, wartości: suma(wartosc)
przychod_godzina_kategoria = ...  # Replace ... with your code

przychod_godzina_kategoria.orderBy("godzina").show()

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 2 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: wyciągamy godzinę z timestampu i liczymy wartość transakcji (cena * ilosc)
transakcje_z_godzina = transakcje.withColumn("godzina", F.hour("data_czas")).withColumn(
    "wartosc", F.col("cena") * F.col("ilosc")
)

# Krok 2: pivot — wiersze to godziny, kolumny to kategorie produktów, wartości to suma przychodu
przychod_godzina_kategoria = (
    transakcje_z_godzina
    .groupBy("godzina")
    .pivot("kategoria")
    .agg(F.round(F.sum("wartosc"), 2))
)

przychod_godzina_kategoria.orderBy("godzina").show()


In [ ]:
# @title ✅ Test — Ćwiczenie 2 (Run to check your answer)

try:
    assert przychod_godzina_kategoria is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    assert "godzina" in przychod_godzina_kategoria.columns, "❌ Brakuje kolumny 'godzina'"
    kategorie_oczekiwane = {"kawa", "herbata", "przekaska", "inne"}
    assert kategorie_oczekiwane.issubset(set(przychod_godzina_kategoria.columns)), \
        "❌ Brakuje kolumn dla wszystkich kategorii po pivocie"
    liczba_godzin = przychod_godzina_kategoria.select("godzina").distinct().count()
    assert liczba_godzin > 1, "❌ Powinno być więcej niż jedna wartość godziny"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 3: Wzrost przychodu miesiąc do miesiąca

**Trudność:** ⭐⭐⭐

Dla całej sieci kawiarni policz przychód **per miesiąc**, a następnie dodaj kolumnę pokazującą **procentowy wzrost** względem poprzedniego miesiąca.

**Oczekiwany rezultat:** DataFrame `wzrost_mom` z kolumnami `miesiac`, `przychod`, `przychod_poprzedni_miesiac`, `wzrost_procent`, posortowany rosnąco po `miesiac`.

> 💡 **Hint:** Wyciągnij miesiąc przez `F.date_format("data_czas", "yyyy-MM")`, zsumuj przychód, a potem użyj `F.lag()` na oknie posortowanym po miesiącu (bez `partitionBy`, bo interesuje nas cała sieć).

In [ ]:
# ✏️ Ćwiczenie 3: Wzrost przychodu miesiąc do miesiąca

# TODO: dodaj kolumnę 'miesiac' (format 'yyyy-MM') oraz 'wartosc' (cena * ilosc)
transakcje_z_miesiacem = ...  # Replace ... with your code

# TODO: zagreguj przychód per miesiąc
przychod_miesieczny = ...  # Replace ... with your code

# TODO: zdefiniuj okno posortowane po miesiącu (bez partitionBy)
okno_czas = ...  # Replace ... with your code

# TODO: dodaj kolumnę z przychodem z poprzedniego miesiąca (F.lag) oraz wzrost procentowy
wzrost_mom = ...  # Replace ... with your code

wzrost_mom.orderBy("miesiac").show()

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 3 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: wyciągamy miesiąc (yyyy-MM) z timestampu i liczymy wartość każdej transakcji
transakcje_z_miesiacem = transakcje.withColumn(
    "miesiac", F.date_format("data_czas", "yyyy-MM")
).withColumn("wartosc", F.col("cena") * F.col("ilosc"))

# Krok 2: sumujemy przychód całej sieci kawiarni per miesiąc
przychod_miesieczny = (
    transakcje_z_miesiacem
    .groupBy("miesiac")
    .agg(F.round(F.sum("wartosc"), 2).alias("przychod"))
)

# Krok 3: jedno globalne okno posortowane po miesiącu (bez partitionBy — interesuje nas cała sieć)
okno_czas = Window.orderBy("miesiac")

# Krok 4: F.lag pobiera przychód z poprzedniego wiersza (poprzedniego miesiąca), liczymy wzrost %
wzrost_mom = (
    przychod_miesieczny
    .withColumn("przychod_poprzedni_miesiac", F.lag("przychod").over(okno_czas))
    .withColumn(
        "wzrost_procent",
        F.round(
            (F.col("przychod") - F.col("przychod_poprzedni_miesiac"))
            / F.col("przychod_poprzedni_miesiac") * 100,
            2,
        ),
    )
)

wzrost_mom.orderBy("miesiac").show()


In [ ]:
# @title ✅ Test — Ćwiczenie 3 (Run to check your answer)

try:
    assert wzrost_mom is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    wymagane_kolumny = {"miesiac", "przychod", "przychod_poprzedni_miesiac", "wzrost_procent"}
    assert wymagane_kolumny.issubset(set(wzrost_mom.columns)), \
        f"❌ Brakuje kolumn: {wymagane_kolumny - set(wzrost_mom.columns)}"
    liczba_miesiecy = wzrost_mom.count()
    assert liczba_miesiecy >= 10, f"❌ Oczekiwano danych z ~12 miesięcy, otrzymano {liczba_miesiecy}"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 4: Join z tabelą `sklepy` — przychód per region

**Trudność:** ⭐⭐

Tabela `sklepy` zawiera informacje o regionie każdego sklepu. Połącz `transakcje` z `sklepy` i oblicz łączny przychód **per region**.

**Oczekiwany rezultat:** DataFrame `przychod_region` z kolumnami `region`, `przychod`, posortowany malejąco po przychodzie.

> 💡 **Hint:** Tabela `sklepy` ma tylko 15 wierszy — to świetny kandydat na `F.broadcast()`!

In [ ]:
# ✏️ Ćwiczenie 4: Join z tabelą sklepy — przychód per region

# TODO: połącz transakcje ze sklepami po kolumnie 'sklep' (użyj F.broadcast na sklepy)
transakcje_ze_sklepami = ...  # Replace ... with your code

# TODO: zagreguj przychód (cena * ilosc) per region i posortuj malejąco
przychod_region = ...  # Replace ... with your code

przychod_region.show()

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 4 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: łączymy transakcje ze sklepami po kolumnie 'sklep'; broadcast, bo 'sklepy' ma tylko 15 wierszy
transakcje_ze_sklepami = transakcje.join(F.broadcast(sklepy), "sklep")

# Krok 2: liczymy przychód (cena * ilosc) per region i sortujemy malejąco
przychod_region = (
    transakcje_ze_sklepami
    .withColumn("wartosc", F.col("cena") * F.col("ilosc"))
    .groupBy("region")
    .agg(F.round(F.sum("wartosc"), 2).alias("przychod"))
    .orderBy(F.col("przychod").desc())
)

przychod_region.show()


In [ ]:
# @title ✅ Test — Ćwiczenie 4 (Run to check your answer)

try:
    assert przychod_region is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    assert "region" in przychod_region.columns, "❌ Brakuje kolumny 'region'"
    liczba_regionow = przychod_region.count()
    assert liczba_regionow == 5, f"❌ Oczekiwano 5 regionów, otrzymano {liczba_regionow}"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 5: Najpopularniejsza metoda płatności per miasto

**Trudność:** ⭐⭐⭐

Dla każdego miasta znajdź **najczęściej używaną** metodę płatności (najwyższa liczba transakcji).

**Oczekiwany rezultat:** DataFrame `top_metoda_platnosci` z kolumnami `miasto`, `metoda_platnosci`, `liczba_transakcji` — dokładnie jeden wiersz na miasto.

> 💡 **Hint:** Podobnie jak w Ćwiczeniu 1 — zlicz transakcje per `miasto` + `metoda_platnosci`, a potem użyj okna z `F.row_number()`.

In [ ]:
# ✏️ Ćwiczenie 5: Najpopularniejsza metoda płatności per miasto

# TODO: policz liczbę transakcji per miasto + metoda_platnosci
liczba_transakcji_miasto_metoda = ...  # Replace ... with your code

# TODO: zdefiniuj okno partycjonowane po mieście, posortowane malejąco po liczbie transakcji
okno_miasto = ...  # Replace ... with your code

# TODO: dodaj ranking i zostaw tylko ranga == 1
top_metoda_platnosci = ...  # Replace ... with your code

top_metoda_platnosci.show()

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 5 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: liczymy liczbę transakcji per miasto + metoda_platnosci
liczba_transakcji_miasto_metoda = (
    transakcje
    .groupBy("miasto", "metoda_platnosci")
    .agg(F.count("transakcja_id").alias("liczba_transakcji"))
)

# Krok 2: okno partycjonowane po mieście, posortowane malejąco wg liczby transakcji
okno_miasto = Window.partitionBy("miasto").orderBy(F.desc("liczba_transakcji"))

# Krok 3: row_number zapewnia dokładnie jeden wiersz na miasto, nawet przy remisach
top_metoda_platnosci = (
    liczba_transakcji_miasto_metoda
    .withColumn("ranking", F.row_number().over(okno_miasto))
    .filter(F.col("ranking") == 1)
    .select("miasto", "metoda_platnosci", "liczba_transakcji")
)

top_metoda_platnosci.show()


In [ ]:
# @title ✅ Test — Ćwiczenie 5 (Run to check your answer)

try:
    assert top_metoda_platnosci is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    liczba_miast = top_metoda_platnosci.select("miasto").distinct().count()
    assert liczba_miast == 5, f"❌ Oczekiwano 5 miast, otrzymano {liczba_miast}"
    liczba_wierszy = top_metoda_platnosci.count()
    assert liczba_wierszy == 5, "❌ Powinien być dokładnie jeden wiersz na miasto"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 6: Zapytanie SQL z `HAVING`

**Trudność:** ⭐⭐

Napisz zapytanie `spark.sql()`, które zwraca sklepy (`sklep`), których **łączny przychód przekracza 40 000 PLN**.

**Oczekiwany rezultat:** DataFrame `sklepy_powyzej_progu` z kolumnami `sklep`, `przychod`, posortowany malejąco po przychodzie.

> 💡 **Hint:** `GROUP BY sklep HAVING SUM(cena * ilosc) > 40000` — pamiętaj, że `HAVING` filtruje **po** agregacji, w przeciwieństwie do `WHERE`.

In [ ]:
# ✏️ Ćwiczenie 6: Zapytanie SQL z HAVING

# TODO: napisz zapytanie spark.sql() z GROUP BY ... HAVING
sklepy_powyzej_progu = ...  # Replace ... with your code

sklepy_powyzej_progu.show()

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 6 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: GROUP BY + HAVING filtrują PO agregacji (w przeciwieństwie do WHERE, które filtruje przed)
sklepy_powyzej_progu = spark.sql("""
    SELECT sklep, ROUND(SUM(cena * ilosc), 2) AS przychod
    FROM transakcje
    GROUP BY sklep
    HAVING SUM(cena * ilosc) > 40000
    ORDER BY przychod DESC
""")

sklepy_powyzej_progu.show()


In [ ]:
# @title ✅ Test — Ćwiczenie 6 (Run to check your answer)

try:
    assert sklepy_powyzej_progu is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    assert {"sklep", "przychod"}.issubset(set(sklepy_powyzej_progu.columns)), \
        "❌ Brakuje wymaganych kolumn ('sklep', 'przychod')"
    progi = [wiersz["przychod"] for wiersz in sklepy_powyzej_progu.collect()]
    assert all(p > 40000 for p in progi), "❌ Wszystkie przychody powinny przekraczać 40000"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 7: Filtrowanie zaawansowane — poranny ruch

**Trudność:** ⭐⭐

Zarząd chce wiedzieć, jaki **procent** wszystkich transakcji odbywa się **rano** (godziny 6:00–10:00, kategoria `kawa`, płatność `blik`).

1. Dodaj kolumnę `godzina`
2. Przefiltruj transakcje spełniające **wszystkie trzy** warunki jednocześnie
3. Oblicz procent takich transakcji względem wszystkich transakcji

**Oczekiwany rezultat:** zmienna `procent_porannych_blik` typu `float`.

> 💡 **Hint:** Połącz warunki operatorem `&` w `.filter()`, np. `(F.col("godzina") >= 6) & (F.col("godzina") < 10) & ...`. Pamiętaj o nawiasach wokół każdego warunku!

In [ ]:
# ✏️ Ćwiczenie 7: Filtrowanie zaawansowane — poranny ruch

# TODO: dodaj kolumnę 'godzina'
transakcje_godzina7 = ...  # Replace ... with your code

# TODO: przefiltruj: godzina 6-10, kategoria == 'kawa', metoda_platnosci == 'blik'
poranne_blik = ...  # Replace ... with your code

# TODO: oblicz procent (float) takich transakcji względem wszystkich transakcji
procent_porannych_blik = ...  # Replace ... with your code

print(f"Procent porannych transakcji kawa+blik: {procent_porannych_blik:.2f}%")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 7 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: dodajemy kolumnę 'godzina' wyciągniętą z timestampu
transakcje_godzina7 = transakcje.withColumn("godzina", F.hour("data_czas"))

# Krok 2: łączymy trzy warunki operatorem & — każdy warunek w osobnych nawiasach!
poranne_blik = transakcje_godzina7.filter(
    (F.col("godzina") >= 6)
    & (F.col("godzina") < 10)
    & (F.col("kategoria") == "kawa")
    & (F.col("metoda_platnosci") == "blik")
)

# Krok 3: procent = liczba transakcji spełniających warunki / liczba wszystkich transakcji * 100
procent_porannych_blik = float(poranne_blik.count() / transakcje.count() * 100)

print(f"Procent porannych transakcji kawa+blik: {procent_porannych_blik:.2f}%")


In [ ]:
# @title ✅ Test — Ćwiczenie 7 (Run to check your answer)

try:
    assert procent_porannych_blik is not ..., "❌ Nie ukończyłeś/aś jeszcze zadania"
    assert isinstance(procent_porannych_blik, float), "❌ Wynik powinien być typu float"
    assert 0.0 <= procent_porannych_blik <= 100.0, "❌ Procent powinien być między 0 a 100"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## 🏆 Bonus Challenge (Optional): Optymalizacja broadcast join

Dla chętnych! 🌟

1. Wykonaj join `transakcje` z `sklepy` **bez** `F.broadcast()` i wywołaj `.explain()` na wyniku
2. Wykonaj ten sam join **z** `F.broadcast(sklepy)` i również wywołaj `.explain()`
3. Porównaj oba plany wykonania — znajdź w planie z broadcastem krok `BroadcastExchange` / `BroadcastHashJoin`
4. Napisz w komórce Markdown (poniżej), jaką różnicę zauważyłeś/aś w planach wykonania

> 💡 Nawet jeśli Spark **sam** zdecydował się na broadcast (bo `sklepy` jest małą tabelą), warto zobaczyć to w planie na własne oczy!

In [ ]:
# 🏆 Bonus: porównanie planów wykonania z i bez broadcast

print("=== BEZ jawnego broadcast ===")
transakcje.join(sklepy, "sklep").explain()

print("\n=== Z jawnym F.broadcast() ===")
transakcje.join(F.broadcast(sklepy), "sklep").explain()

*(Twoje obserwacje na temat różnicy w planach wykonania — wpisz tutaj)*

---
## 📝 Podsumowanie

Świetna robota! 🎉 Oto co przećwiczyłeś/aś w tym laboratorium:

- ✅ Rejestrowanie tymczasowych widoków i pisanie zapytań `spark.sql()`
- ✅ Agregacje (`groupBy`, `agg`, `pivot`) i ich odpowiedniki SQL
- ✅ Window functions (`row_number`, `lag`) do rankingów i analizy trendów czasowych
- ✅ Joiny z tabelą wymiarową i optymalizacja przez `broadcast()`
- ✅ Filtrowanie z wieloma warunkami jednocześnie

### 📚 Dalsze materiały
- [Spark SQL Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)
- [PySpark Window Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html)

### 🔮 Co dalej?
W **Części B** (poniżej, ten sam notebook) poznasz **zaawansowane window functions** i **optymalizację zapytań** (Catalyst, `explain()`, caching) — czas zamienić się w prawdziwych baristów danych! ☕📊

---

# 🅱️ Część B: Window Functions — Wypożyczalnia rowerów miejskich 🚲

Analiza ~200 000 przejazdów wypożyczalni rowerów miejskich przy pomocy window functions. Ta część generuje **własny, niezależny** zbiór danych (nie korzysta z danych Części A).


In [ ]:
# @title 🔧 Setup — uruchom tę komórkę jako pierwszą!

!pip install -q pyspark

import random
import time
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, TimestampType, DoubleType
)

random.seed(42)  # 👈 powtarzalność wyników

spark = (
    SparkSession.builder
    .appName("Cwiczenie6-RoweryMiejskie")
    .master("local[*]")
    .getOrCreate()
)

print("✅ Setup zakończony! Zaczynamy 🚀")

## 📖 Wprowadzenie — window functions w pigułce

`groupBy()` **redukuje** liczbę wierszy do jednego per grupa. **Window function** dodaje nową kolumnę obliczoną na podstawie "okna" powiązanych wierszy — bez utraty żadnego wiersza.

| Koncept | Opis | Przykład |
|---------|------|----------|
| `Window.partitionBy(col)` | Dzieli dane na grupy (jak `groupBy`, ale bez redukcji) | `Window.partitionBy("stacja")` |
| `Window.orderBy(col)` | Kolejność wierszy wewnątrz partycji | `.orderBy("czas_startu")` |
| `F.rank()` / `F.row_number()` | Ranking (z lukami / bez luk przy remisach) | `F.rank().over(okno)` |
| `F.lag()` / `F.lead()` | Wartość z poprzedniego / następnego wiersza | `F.lag("kolumna", 1).over(okno)` |
| `.rowsBetween(a, b)` | Zakres okna względem bieżącego wiersza | `rowsBetween(Window.unboundedPreceding, Window.currentRow)` |

Dziś pracujemy na **syntetycznym zbiorze ~200 000 przejazdów** rowerów miejskich w trzech polskich miastach 🚲🇵🇱.

In [ ]:
# @title 📊 Generowanie danych — wypożyczenia rowerów miejskich

CITIES = ["Warszawa", "Kraków", "Poznań"]
STATIONS_PER_CITY = 8
N_TRIPS = 200_000

stacje = [
    (miasto, f"{miasto}-Stacja-{i:02d}")
    for miasto in CITIES
    for i in range(1, STATIONS_PER_CITY + 1)
]
stacje_per_miasto = {
    miasto: [nazwa for m, nazwa in stacje if m == miasto] for miasto in CITIES
}

POCZATEK_OKRESU = datetime(2026, 8, 1)


def losowy_czas_startu() -> datetime:
    losowy_dzien = random.randint(0, 90)
    losowa_godzina = random.randint(5, 23)
    losowa_minuta = random.randint(0, 59)
    return POCZATEK_OKRESU + timedelta(
        days=losowy_dzien, hours=losowa_godzina, minutes=losowa_minuta
    )


wiersze = []
for trip_id in range(1, N_TRIPS + 1):
    miasto = random.choice(CITIES)
    stacja_startowa = random.choice(stacje_per_miasto[miasto])
    stacja_koncowa = random.choice(stacje_per_miasto[miasto])

    dystans_km = round(random.uniform(0.3, 12.0), 2)
    predkosc_kmh = random.uniform(9.0, 16.0)
    szum_min = random.uniform(-2.0, 5.0)
    czas_trwania_min = max(1.0, round(dystans_km / predkosc_kmh * 60 + szum_min, 1))

    czas_startu = losowy_czas_startu()
    czas_konca = czas_startu + timedelta(minutes=czas_trwania_min)

    typ_uzytkownika = random.choices(["abonament", "jednorazowy"], weights=[0.7, 0.3])[0]

    wiersze.append((
        trip_id, miasto, stacja_startowa, stacja_koncowa,
        czas_startu, czas_konca, czas_trwania_min,
        typ_uzytkownika, dystans_km,
    ))

print(f"Wygenerowano {len(wiersze):,} przejazdów")

schema = StructType([
    StructField("trip_id", IntegerType(), False),
    StructField("city", StringType(), False),
    StructField("start_station", StringType(), False),
    StructField("end_station", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("duration_min", DoubleType(), False),
    StructField("user_type", StringType(), False),
    StructField("distance_km", DoubleType(), False),
])

trips = spark.createDataFrame(wiersze, schema=schema)
trips = trips.withColumn("trip_date", F.to_date("start_time"))
trips.cache()

print(f"📊 Liczba wierszy: {trips.count():,}")
trips.printSchema()
trips.show(5, truncate=False)

## 🎓 Przykład prowadzony — ranking stacji wg liczby przejazdów

**Zadanie:** Dla każdego miasta chcemy ranking stacji startowych wg liczby wypożyczeń — stacja #1 to najbardziej ruchliwa stacja w danym mieście.

In [ ]:
# @title 🎓 Przykład prowadzony — uruchom i przeanalizuj

# Krok 1: liczba przejazdów per stacja startowa (w ramach miasta)
wolumen_stacji = trips.groupBy("city", "start_station").agg(
    F.count("trip_id").alias("liczba_przejazdow")
)

# Krok 2: okno — partycja per miasto, sortowanie malejąco wg liczby przejazdów
okno_rankingu = Window.partitionBy("city").orderBy(F.desc("liczba_przejazdow"))

# Krok 3: nadanie rankingu
ranking_stacji = wolumen_stacji.withColumn(
    "ranking", F.rank().over(okno_rankingu)  # 👈 window function w akcji!
)

ranking_stacji.orderBy("city", "ranking").show(24, truncate=False)

---
## ✏️ Ćwiczenie 1: Ranking miast wg łącznej liczby przejazdów

**Trudność:** ⭐

Policz łączną liczbę przejazdów w każdym mieście, a następnie nadaj każdemu miastu **ranking** (bez `partitionBy` — jedno globalne okno!). Miasto z największą liczbą przejazdów powinno mieć `ranking == 1`.

> 💡 **Hint:** `Window.orderBy(...)` bez `partitionBy` tworzy jedno okno obejmujące cały DataFrame.

In [ ]:
# ✏️ Ćwiczenie 1: Ranking miast

liczba_przejazdow_miasto = trips.groupBy("city").agg(
    F.count("trip_id").alias("liczba_przejazdow")
)

# TODO: zdefiniuj okno sortujące malejąco wg liczby przejazdów (bez partitionBy)
okno_miast = ...

# TODO: nadaj ranking używając F.rank()
ranking_miast = liczba_przejazdow_miasto.withColumn(
    "ranking", ...
)

ranking_miast.orderBy("ranking").show()

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 1 (Część B) (kliknij, aby zobaczyć krok po kroku)

# Krok 1: okno bez partitionBy obejmuje cały DataFrame jako jedną globalną grupę
okno_miast = Window.orderBy(F.desc("liczba_przejazdow"))

# Krok 2: F.rank() nadaje ranking 1 miastu z największą liczbą przejazdów
ranking_miast = liczba_przejazdow_miasto.withColumn(
    "ranking", F.rank().over(okno_miast)
)

ranking_miast.orderBy("ranking").show()


In [ ]:
# @title ✅ Test — Ćwiczenie 1

try:
    assert "ranking" in ranking_miast.columns, "❌ Brakuje kolumny 'ranking'"
    wynik = ranking_miast.orderBy("ranking").collect()
    assert len(wynik) == 3, f"❌ Oczekiwano 3 miast, jest {len(wynik)}"
    assert wynik[0]["ranking"] == 1, "❌ Pierwsze miejsce powinno mieć ranking 1"
    print("✅ Świetnie! Ranking miast gotowy 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 2: Suma narastająca (running total) przejazdów per stacja

**Trudność:** ⭐⭐

Dla każdej stacji policz **dzienną** liczbę przejazdów, a następnie dodaj kolumnę z **sumą narastającą** (running total) w czasie — używając `rowsBetween(Window.unboundedPreceding, Window.currentRow)`.

**Oczekiwany wynik:** Kolumna `suma_narastajaca` powinna **nigdy nie maleć** w obrębie jednej stacji, a jej ostatnia wartość powinna równać się sumie wszystkich dziennych przejazdów tej stacji.

In [ ]:
# ✏️ Ćwiczenie 2: Running total

dzienne_przejazdy = trips.groupBy("start_station", "trip_date").agg(
    F.count("trip_id").alias("liczba_przejazdow_dziennie")
)

# TODO: zdefiniuj okno: partycja per stacja, sortowanie po dacie, zakres od początku do bieżącego wiersza
okno_narastajace = (
    Window.partitionBy("start_station")
    .orderBy("trip_date")
    .rowsBetween(...)  # TODO: (Window.unboundedPreceding, Window.currentRow)
)

# TODO: dodaj kolumnę z sumą narastającą, używając F.sum(...).over(...)
dzienne_z_suma = dzienne_przejazdy.withColumn(
    "suma_narastajaca", ...
)

dzienne_z_suma.orderBy("start_station", "trip_date").show(10)

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 2 (Część B) (kliknij, aby zobaczyć krok po kroku)

# Krok 1: okno partycjonowane po stacji, posortowane po dacie, zakres od początku partycji do bieżącego wiersza
okno_narastajace = (
    Window.partitionBy("start_station")
    .orderBy("trip_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Krok 2: suma narastająca = suma liczby przejazdów od pierwszego dnia stacji do bieżącego dnia
dzienne_z_suma = dzienne_przejazdy.withColumn(
    "suma_narastajaca", F.sum("liczba_przejazdow_dziennie").over(okno_narastajace)
)

dzienne_z_suma.orderBy("start_station", "trip_date").show(10)


In [ ]:
# @title ✅ Test — Ćwiczenie 2

try:
    przykladowa_stacja = dzienne_z_suma.select("start_station").first()["start_station"]
    pdf = (
        dzienne_z_suma.filter(F.col("start_station") == przykladowa_stacja)
        .orderBy("trip_date")
        .toPandas()
    )
    assert pdf["suma_narastajaca"].is_monotonic_increasing, (
        "❌ Suma narastająca powinna rosnąć (lub być stała) w czasie"
    )
    assert pdf["suma_narastajaca"].iloc[-1] == pdf["liczba_przejazdow_dziennie"].sum(), (
        "❌ Ostatnia wartość powinna równać się sumie wszystkich dni tej stacji"
    )
    print("✅ Running total działa poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 3: Przerwa czasowa między kolejnymi przejazdami (`lag`)

**Trudność:** ⭐⭐

Dla każdej stacji startowej policz, ile minut minęło od **poprzedniego** przejazdu rozpoczętego z tej samej stacji (`lag`). Pierwszy przejazd ze stacji nie ma poprzednika — jego przerwa powinna być `null`.

> 💡 **Hint:** Odejmij `unix timestamp` obu kolumn (`.cast("long")`) i podziel przez 60, żeby dostać minuty.

In [ ]:
# ✏️ Ćwiczenie 3: lag — przerwa czasowa

okno_czasowe = Window.partitionBy("start_station").orderBy("start_time")

# TODO: dodaj kolumnę 'poprzedni_start' z czasem poprzedniego przejazdu (F.lag)
przejazdy_z_przerwa = trips.withColumn(
    "poprzedni_start", ...
)

przejazdy_z_przerwa = przejazdy_z_przerwa.withColumn(
    "przerwa_minuty",
    (F.col("start_time").cast("long") - F.col("poprzedni_start").cast("long")) / 60
)

przejazdy_z_przerwa.select(
    "start_station", "start_time", "poprzedni_start", "przerwa_minuty"
).orderBy("start_station", "start_time").show(10)

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 3 (Część B) (kliknij, aby zobaczyć krok po kroku)

# Krok 1: F.lag pobiera wartość 'start_time' z poprzedniego wiersza tej samej partycji (stacji);
# dla pierwszego wiersza partycji lag zwraca null — i tak właśnie powinno być
przejazdy_z_przerwa = trips.withColumn(
    "poprzedni_start", F.lag("start_time", 1).over(okno_czasowe)
)

przejazdy_z_przerwa = przejazdy_z_przerwa.withColumn(
    "przerwa_minuty",
    (F.col("start_time").cast("long") - F.col("poprzedni_start").cast("long")) / 60
)

przejazdy_z_przerwa.select(
    "start_station", "start_time", "poprzedni_start", "przerwa_minuty"
).orderBy("start_station", "start_time").show(10)


In [ ]:
# @title ✅ Test — Ćwiczenie 3

try:
    pierwszy_wiersz = przejazdy_z_przerwa.orderBy("start_station", "start_time").first()
    assert pierwszy_wiersz["poprzedni_start"] is None, (
        "❌ Pierwszy przejazd ze stacji nie powinien mieć poprzednika (oczekiwano null)"
    )
    probka = przejazdy_z_przerwa.filter(F.col("poprzedni_start").isNotNull()).limit(1000).collect()
    assert all(w["przerwa_minuty"] >= 0 for w in probka), "❌ Przerwa czasowa nie może być ujemna"
    print("✅ Obliczono przerwy czasowe poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 4: Top-3 najbardziej ruchliwe stacje końcowe per miasto

**Trudność:** ⭐⭐

Znajdź **dokładnie 3** stacje końcowe (`end_station`) z największą liczbą przyjazdów w każdym mieście. Użyj `F.row_number()` — gwarantuje dokładnie 3 wiersze na miasto, nawet przy remisach.

> ⚠️ Gdybyś użył/a `F.rank()`, przy remisie mógłbyś dostać więcej niż 3 wiersze!

In [ ]:
# ✏️ Ćwiczenie 4: Top-3 stacje końcowe per miasto

liczba_przyjazdow = trips.groupBy("city", "end_station").agg(
    F.count("trip_id").alias("liczba_przyjazdow")
)

okno_top = Window.partitionBy("city").orderBy(F.desc("liczba_przyjazdow"))

# TODO: dodaj kolumnę 'pozycja' używając F.row_number()
top3 = liczba_przyjazdow.withColumn(
    "pozycja", ...
)

# TODO: przefiltruj tylko wiersze z pozycja <= 3
top3 = top3.filter(...)

top3.orderBy("city", "pozycja").show(9, truncate=False)

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 4 (Część B) (kliknij, aby zobaczyć krok po kroku)

# Krok 1: F.row_number() gwarantuje unikalną, kolejną pozycję nawet przy remisach (inaczej niż F.rank())
top3 = liczba_przyjazdow.withColumn(
    "pozycja", F.row_number().over(okno_top)
)

# Krok 2: zostawiamy tylko 3 pierwsze pozycje w każdym mieście
top3 = top3.filter(F.col("pozycja") <= 3)

top3.orderBy("city", "pozycja").show(9, truncate=False)


In [ ]:
# @title ✅ Test — Ćwiczenie 4

try:
    wyniki_per_miasto = top3.groupBy("city").count().collect()
    assert all(row["count"] == 3 for row in wyniki_per_miasto), (
        "❌ Każde miasto powinno mieć dokładnie 3 stacje w top3"
    )
    pierwsze_miasto = CITIES[0]
    pozycje = sorted(
        row["pozycja"] for row in top3.filter(F.col("city") == pierwsze_miasto).collect()
    )
    assert pozycje == [1, 2, 3], f"❌ Oczekiwano pozycji [1, 2, 3], jest {pozycje}"
    print("✅ Top-3 stacje per miasto obliczone poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 5: Średnia krocząca czasu trwania przejazdu

**Trudność:** ⭐⭐⭐

Dla każdej stacji startowej policz **średnią kroczącą** (moving average) `duration_min` z bieżącego i 2 poprzednich przejazdów (`rowsBetween(-2, 0)`), posortowanych po `start_time`.

**Oczekiwany wynik:** Dla pierwszego przejazdu ze stacji średnia krocząca powinna równać się jego własnemu `duration_min` (bo nie ma jeszcze 2 poprzednich).

In [ ]:
# ✏️ Ćwiczenie 5: Średnia krocząca

# TODO: zdefiniuj okno z rowsBetween(-2, 0) — bieżący wiersz + 2 poprzednie
okno_kroczace = (
    Window.partitionBy("start_station")
    .orderBy("start_time")
    .rowsBetween(...)
)

# TODO: dodaj kolumnę 'srednia_kroczaca_min' używając F.avg("duration_min").over(...)
z_srednia_kroczaca = trips.withColumn(
    "srednia_kroczaca_min", ...
)

z_srednia_kroczaca.select(
    "start_station", "start_time", "duration_min", "srednia_kroczaca_min"
).orderBy("start_station", "start_time").show(10)

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 5 (Część B) (kliknij, aby zobaczyć krok po kroku)

# Krok 1: rowsBetween(-2, 0) bierze bieżący wiersz oraz dwa poprzednie w ramach partycji (stacji)
okno_kroczace = (
    Window.partitionBy("start_station")
    .orderBy("start_time")
    .rowsBetween(-2, 0)
)

# Krok 2: F.avg liczy średnią duration_min z wierszy objętych oknem — dla 1. przejazdu stacji
# okno zawiera tylko jego samego, więc średnia krocząca = jego własny czas trwania
z_srednia_kroczaca = trips.withColumn(
    "srednia_kroczaca_min", F.avg("duration_min").over(okno_kroczace)
)

z_srednia_kroczaca.select(
    "start_station", "start_time", "duration_min", "srednia_kroczaca_min"
).orderBy("start_station", "start_time").show(10)


In [ ]:
# @title ✅ Test — Ćwiczenie 5

try:
    pierwszy = z_srednia_kroczaca.orderBy("start_station", "start_time").first()
    assert abs(pierwszy["srednia_kroczaca_min"] - pierwszy["duration_min"]) < 0.01, (
        "❌ Dla pierwszego przejazdu średnia krocząca powinna równać się jego własnemu czasowi trwania"
    )
    print("✅ Średnia krocząca policzona poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 6: Czytanie planu zapytania (`.explain()`)

**Trudność:** ⭐⭐

Uruchom poniższe zapytanie i przeanalizuj wypisany `Physical Plan`. Na tej podstawie odpowiedz (wpisując `True`/`False`) na dwa pytania.

> 💡 **Hint:** Shuffle w planie fizycznym objawia się węzłem `Exchange`. Predicate pushdown oznacza, że `Filter` pojawia się **przed** (czyli fizycznie bliżej źródła danych niż) `Aggregate`/`HashAggregate`.

In [ ]:
# ✏️ Ćwiczenie 6: Czytanie planu zapytania

zapytanie = (
    trips.filter(F.col("user_type") == "abonament")
    .groupBy("city")
    .agg(F.sum("distance_km").alias("suma_dystansu"))
    .orderBy(F.desc("suma_dystansu"))
)

zapytanie.explain(True)

# TODO: na podstawie wypisanego Physical Plan odpowiedz True/False
czy_jest_shuffle = ...  # Czy w Physical Plan widzisz węzeł "Exchange"?
czy_filtr_jest_wczesnie = ...  # Czy "Filter" pojawia się przed "HashAggregate" w planie fizycznym?

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 6 (Część B) (kliknij, aby zobaczyć krok po kroku)

# Krok 1: groupBy (HashAggregate) oraz orderBy wymagają przetasowania danych między partycjami —
# w Physical Plan widać to jako węzeł "Exchange" (shuffle)
czy_jest_shuffle = True

# Krok 2: Catalyst Optimizer stosuje predicate pushdown — filtr wykonuje się jak najwcześniej,
# więc w planie fizycznym "Filter" znajduje się przed (bliżej źródła danych niż) "HashAggregate"
czy_filtr_jest_wczesnie = True

print(f"Czy jest shuffle: {czy_jest_shuffle}")
print(f"Czy filtr jest wcześnie: {czy_filtr_jest_wczesnie}")


In [ ]:
# @title ✅ Test — Ćwiczenie 6

try:
    assert czy_jest_shuffle is True, (
        "❌ groupBy + orderBy generują shuffle — sprawdź plan jeszcze raz (szukaj 'Exchange')"
    )
    assert czy_filtr_jest_wczesnie is True, (
        "❌ Catalyst powinien zepchnąć Filter przed Aggregate (predicate pushdown)"
    )
    print("✅ Dobrze przeczytany plan zapytania! 🎉")
except AssertionError as e:
    print(str(e))
    print("Uruchom jeszcze raz zapytanie.explain(True) i przyjrzyj się dokładnie 💪")

---
## ✏️ Ćwiczenie 7: `repartition()` vs `coalesce()` — porównanie czasu

**Trudność:** ⭐⭐⭐

Zmierz czas wykonania zmniejszenia liczby partycji do 4, używając osobno `repartition(4)` i `coalesce(4)`. Zapisz zmierzone czasy i liczby partycji po każdej operacji.

> 💡 **Hint:** `repartition()` zawsze robi pełny shuffle (drożej), `coalesce()` łączy sąsiadujące partycje bez pełnego shuffla (zwykle taniej).

In [ ]:
# ✏️ Ćwiczenie 7: repartition vs coalesce

print(f"Liczba partycji na starcie: {trips.rdd.getNumPartitions()}")

start = time.time()
# TODO: użyj trips.repartition(4)
trips_repartition = ...
trips_repartition.count()  # akcja wymuszająca wykonanie
czas_repartition = time.time() - start

start = time.time()
# TODO: użyj trips.coalesce(4)
trips_coalesce = ...
trips_coalesce.count()
czas_coalesce = time.time() - start

print(f"repartition(4): {czas_repartition:.2f}s")
print(f"coalesce(4):    {czas_coalesce:.2f}s")

# TODO: uzupełnij liczbę partycji po każdej operacji
liczba_partycji_repartition = ...
liczba_partycji_coalesce = ...

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 7 (Część B) (kliknij, aby zobaczyć krok po kroku)

print(f"Liczba partycji na starcie: {trips.rdd.getNumPartitions()}")

start = time.time()
# Krok 1: repartition(4) wykonuje PEŁNY shuffle, żeby równomiernie rozłożyć dane na 4 partycje
trips_repartition = trips.repartition(4)
trips_repartition.count()  # akcja wymuszająca wykonanie
czas_repartition = time.time() - start

start = time.time()
# Krok 2: coalesce(4) łączy sąsiadujące partycje BEZ pełnego shuffla — zwykle szybsze przy zmniejszaniu
trips_coalesce = trips.coalesce(4)
trips_coalesce.count()
czas_coalesce = time.time() - start

print(f"repartition(4): {czas_repartition:.2f}s")
print(f"coalesce(4):    {czas_coalesce:.2f}s")

# Krok 3: odczytujemy faktyczną liczbę partycji po każdej operacji
liczba_partycji_repartition = trips_repartition.rdd.getNumPartitions()
liczba_partycji_coalesce = trips_coalesce.rdd.getNumPartitions()


In [ ]:
# @title ✅ Test — Ćwiczenie 7

try:
    assert liczba_partycji_repartition == 4, "❌ repartition(4) powinno dać 4 partycje"
    assert liczba_partycji_coalesce == 4, "❌ coalesce(4) powinno dać 4 partycje"
    print("✅ Sprawdzone! coalesce zwykle jest szybszy przy zmniejszaniu partycji, bo unika pełnego shuffla 🚀")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## 🏆 Bonus Challenge: Wykrywanie potencjalnego "rebalancingu" rowerów

Operator wypożyczalni czasem ręcznie **przewozi rowery** między stacjami ("rebalancing"), żeby np. uzupełnić pustą stację w centrum. Możemy to pośrednio wykryć: jeśli między **przyjazdem** roweru na stację a **kolejnym odjazdem** z tej samej stacji mija bardzo dużo czasu (> 240 minut), to podejrzanie długo ten rower tam "stał" — a scenariusz interwencji operatora jest prawdopodobny.

**Zadanie:**
1. Zbuduj DataFrame zdarzeń łączący przyjazdy (`end_station`, `end_time`) i odjazdy (`start_station`, `start_time`) dla każdej stacji
2. Użyj `F.lead()`, żeby dla każdego zdarzenia znać **następne** zdarzenie na tej samej stacji
3. Oznacz jako anomalię wiersze, gdzie: bieżące zdarzenie to `"przyjazd"`, następne to `"odjazd"`, a przerwa > 240 minut

In [ ]:
# 🏆 Bonus: wykrywanie anomalii rebalancingu

przyjazdy = trips.select(
    F.col("end_station").alias("stacja"),
    F.col("end_time").alias("czas_zdarzenia"),
    F.lit("przyjazd").alias("typ_zdarzenia"),
)
odjazdy = trips.select(
    F.col("start_station").alias("stacja"),
    F.col("start_time").alias("czas_zdarzenia"),
    F.lit("odjazd").alias("typ_zdarzenia"),
)
zdarzenia = przyjazdy.unionByName(odjazdy)

okno_zdarzen = Window.partitionBy("stacja").orderBy("czas_zdarzenia")

zdarzenia_z_nastepnym = (
    zdarzenia
    .withColumn("nastepny_czas", F.lead("czas_zdarzenia", 1).over(okno_zdarzen))
    .withColumn("nastepny_typ", F.lead("typ_zdarzenia", 1).over(okno_zdarzen))
    .withColumn(
        "przerwa_minut",
        (F.col("nastepny_czas").cast("long") - F.col("czas_zdarzenia").cast("long")) / 60
    )
)

# TODO: dopisz warunek filtrowania — przyjazd -> odjazd z przerwą > 240 minut
anomalie_rebalancingu = zdarzenia_z_nastepnym.filter(
    ...
)

anomalie_rebalancingu.select(
    "stacja", "czas_zdarzenia", "nastepny_czas", "przerwa_minut"
).orderBy(F.desc("przerwa_minut")).show(10)

In [ ]:
# @title 💡 Pełne rozwiązanie — Bonus (Część B) (kliknij, aby zobaczyć krok po kroku)

# Warunek anomalii: bieżące zdarzenie to przyjazd, następne to odjazd z tej samej stacji,
# a przerwa między nimi przekracza 240 minut (rower podejrzanie długo stał bezczynnie)
anomalie_rebalancingu = zdarzenia_z_nastepnym.filter(
    (F.col("typ_zdarzenia") == "przyjazd")
    & (F.col("nastepny_typ") == "odjazd")
    & (F.col("przerwa_minut") > 240)
)

anomalie_rebalancingu.select(
    "stacja", "czas_zdarzenia", "nastepny_czas", "przerwa_minut"
).orderBy(F.desc("przerwa_minut")).show(10)


In [ ]:
# @title ✅ Test — Bonus

try:
    kolumny_wymagane = {
        "stacja", "czas_zdarzenia", "typ_zdarzenia",
        "nastepny_czas", "nastepny_typ", "przerwa_minut",
    }
    assert kolumny_wymagane.issubset(set(anomalie_rebalancingu.columns)), (
        "❌ Brakuje wymaganych kolumn w wyniku"
    )
    liczba_anomalii = anomalie_rebalancingu.count()
    if liczba_anomalii > 0:
        przyklad = anomalie_rebalancingu.first()
        assert przyklad["przerwa_minut"] > 240, "❌ Przerwa powinna przekraczać próg 240 minut"
        assert przyklad["typ_zdarzenia"] == "przyjazd", "❌ Bieżące zdarzenie powinno być przyjazdem"
        assert przyklad["nastepny_typ"] == "odjazd", "❌ Następne zdarzenie powinno być odjazdem"
    print(f"🏆 Bonus zaliczony! Znaleziono {liczba_anomalii} potencjalnych zdarzeń rebalancingu")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## 📝 Podsumowanie

Świetna robota! 🎉 W tym ćwiczeniu przećwiczyłeś/aś:

- ✅ Definiowanie okien (`Window.partitionBy()` + `orderBy()`)
- ✅ Rankingi (`rank`, `row_number`) i różnicę między nimi przy remisach
- ✅ Sumy narastające i średnie kroczące (`rowsBetween`)
- ✅ Analizy sekwencyjne "poprzedni/następny" (`lag`, `lead`)
- ✅ Czytanie planu zapytania (`.explain()`) i rozpoznawanie shuffle
- ✅ Różnicę między `repartition()` a `coalesce()` w praktyce

### 📚 Dalsze materiały
- [Window Functions API (PySpark)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html)
- [Catalyst Optimizer — Deep Dive (Databricks)](https://databricks.com/blog/2015/04/13/deep-dive-into-spark-sqls-catalyst-optimizer.html)

### 🔮 Co dalej?
Na **Wykładzie 6** przechodzimy do przygotowania danych do Machine Learning — **Feature Engineering**! Te same dane rowerowe posłużą nam jako materiał do ćwiczeń.